#Notebook 01: Single Cell Data Collection
Downloads GEO Series containing single-cell sequencing data for downstream analysis.

Dataset: GSE179640

Tan Y, Flynn WF, Sivajothi S, Luo D et al. Single-cell analysis of endometriosis reveals a coordinated transcriptional programme driving immunotolerance and angiogenesis across eutopic and ectopic tissues. Nat Cell Biol 2022 Aug;24(8):1306-1318. PMID: 35864314

In [1]:
# -- Installs
!pip -q install anndata scanpy session-info GEOparse pathlib

In [2]:
# -- Imports
import os

from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import session_info
import GEOparse
import scanpy as sc


In [4]:
# -- Mount drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    BASE_DIR = Path("/content/drive/MyDrive/endometriosis-spatial-immune-atlas-main")
except ImportError:
    BASE_DIR = Path("/Users/elenaeyre/Desktop/Science/spatial_tx/endometriosis-spatial-immune-atlas-main")

#from google.colab import drive
#drive.mount('/content/drive', force_remount=True)

In [8]:
# -- Paths
project_dir = BASE_DIR

dataset = "GSE179640"

download_dir = BASE_DIR / "data" / "geo_download"

raw_data_dir = (
    project_dir
    / "data"
    / "raw"
    / dataset
)

interim_data_dir = (
    project_dir
    / "data"
    / "interim"
    / dataset
)

output_file = (
    interim_data_dir
    / "raw.h5ad"
)

raw_data_dir.mkdir(
    parents=True,
    exist_ok=True,
)

interim_data_dir.mkdir(
    parents=True,
    exist_ok=True,
)

In [12]:
# -- Fetch GSE data from GEO
import re
import subprocess

def https_download(url, dest, retries=5):
    # NCBI rejects Python's urllib (HTTP 403) and GEOparse's FTP is unreliable; curl works
    url = url.replace("ftp://", "https://")
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    tmp = dest.with_name(dest.name + ".part")
    subprocess.run(
        ["curl", "-fL", "--retry", str(retries), "--retry-all-errors",
         "--retry-delay", "3", "-o", str(tmp), url],
        check=True,
    )
    tmp.rename(dest)
    return dest

download_dir.mkdir(parents=True, exist_ok=True)
soft_file = download_dir / "GSE179640_family.soft.gz"
https_download(
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE179nnn/GSE179640/soft/GSE179640_family.soft.gz",
    soft_file,
)

gse = GEOparse.get_GEO(filepath=str(soft_file), include_data=True)

for gsm_name, gsm in gse.gsms.items():
    title = re.sub(r"[^\w\-.]+", "_", gsm.metadata["title"][0])
    sample_dir = download_dir / f"Supp_{gsm_name}_{title}"
    for key, urls in gsm.metadata.items():
        if key.startswith("supplementary_file"):
            for url in urls:
                if url.upper() != "NONE":
                    https_download(url, sample_dir / url.split("/")[-1])

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  8690  100  8690    0     0  24130      0 --:--:-- --:--:-- --:--:-- 24138
05-Oct-2026 18:26:01 INFO GEOparse - Parsing /Users/elenaeyre/Desktop/Science/spatial_tx/endometriosis-spatial-immune-atlas-main/data/geo_download/GSE179640_family.soft.gz: 
05-Oct-2026 18:26:01 DEBUG GEOparse - DATABASE: GeoMiame
05-Oct-2026 18:26:01 DEBUG GEOparse - SERIES: GSE179640
05-Oct-2026 18:26:01 DEBUG GEOparse - PLATFORM: GPL24676
05-Oct-2026 18:26:01 DEBUG GEOparse - SAMPLE: GSM6102532
05-Oct-2026 18:26:01 DEBUG GEOparse - SAMPLE: GSM6102533
05-Oct-2026 18:26:01 DEBUG GEOparse - SAMPLE: GSM6102534
05-Oct-2026 18:26:01 DEBUG GEOparse - SAMPLE: GSM6102537
05-Oct-2026 18:26:01 DEBUG GEOparse - SAMPLE: GSM6102540
05-Oct-2026 18:26:01 DEBUG GEOparse - SAMPLE: GSM6102543
05-Oct-2026 18:26:01 DEBUG GEOparse - SAMPLE: GSM6102546
05-Oct-2026 18:26:01

In [13]:
# -- Check downloaded files
downloaded_files = sorted(
    path
    for path in download_dir.rglob("*")
    if path.is_file()
)

print(
    f"Downloaded files: {len(downloaded_files)}"
)

for file in downloaded_files:
    print(file)

Downloaded files: 64
/Users/elenaeyre/Desktop/Science/spatial_tx/endometriosis-spatial-immune-atlas-main/data/geo_download/GSE179640_family.soft.gz
/Users/elenaeyre/Desktop/Science/spatial_tx/endometriosis-spatial-immune-atlas-main/data/geo_download/Supp_GSM6102532_Control_Patient_1_-_Control_Endometrium/GSM6102532_C01_Ctrl_filtered_feature_bc_matrix.h5
/Users/elenaeyre/Desktop/Science/spatial_tx/endometriosis-spatial-immune-atlas-main/data/geo_download/Supp_GSM6102533_Control_Patient_2_-_Control_Endometrium/GSM6102533_C02_Ctrl_filtered_feature_bc_matrix.h5
/Users/elenaeyre/Desktop/Science/spatial_tx/endometriosis-spatial-immune-atlas-main/data/geo_download/Supp_GSM6102534_Control_Patient_3_-_Control_Endometrium/GSM6102534_C03_Ctrl_filtered_feature_bc_matrix.h5
/Users/elenaeyre/Desktop/Science/spatial_tx/endometriosis-spatial-immune-atlas-main/data/geo_download/Supp_GSM6102537_Endometriosis_Patient_1_-_Eutopic/GSM6102537_E01_EuE_filtered_feature_bc_matrix.h5
/Users/elenaeyre/Desktop/Sc

In [14]:
# -- Set up metadata

# samples to exclude
EXCLUDE = {"EOR", "EcPA"}

# metadata map keyed by sample_id extracted from filename
metadata = {
    # controls
    "GSM6102532_C01_Ctrl": {"patient_id": "C01", "tissue_type": "Ctrl",          "condition": "control",         "lesion_site": None},
    "GSM6102533_C02_Ctrl": {"patient_id": "C02", "tissue_type": "Ctrl",          "condition": "control",         "lesion_site": None},
    "GSM6102534_C03_Ctrl": {"patient_id": "C03", "tissue_type": "Ctrl",          "condition": "control",         "lesion_site": None},

    # eutopic
    "GSM6102537_E01_EuE":  {"patient_id": "E01", "tissue_type": "EuE",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102540_E02_EuE":  {"patient_id": "E02", "tissue_type": "EuE",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102543_E03_EuE":  {"patient_id": "E03", "tissue_type": "EuE",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102546_E04_EuE":  {"patient_id": "E04", "tissue_type": "EuE",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102549_E05_EuE":  {"patient_id": "E05", "tissue_type": "EuE",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102551_E06_EuE":  {"patient_id": "E06", "tissue_type": "EuE",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102554_E07_EuE":  {"patient_id": "E07", "tissue_type": "EuE",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102555_E08_EuE":  {"patient_id": "E08", "tissue_type": "EuE",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102560_E09_EuE":  {"patient_id": "E09", "tissue_type": "EuE",          "condition": "endometriosis",   "lesion_site": None},

    # ectopic peritoneal
    "GSM6595248_E01_EcP":  {"patient_id": "E01", "tissue_type": "EcP",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6595250_E02_EcP":  {"patient_id": "E02", "tissue_type": "EcP",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6595252_E03_EcP":  {"patient_id": "E03", "tissue_type": "EcP",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6595254_E04_EcP":  {"patient_id": "E04", "tissue_type": "EcP",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6595256_E05_EcP":  {"patient_id": "E05", "tissue_type": "EcP",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6102550_E06_EcP":  {"patient_id": "E06", "tissue_type": "EcP",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6102553_E07_EcP":  {"patient_id": "E07", "tissue_type": "EcP",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6595258_E09_EcP":  {"patient_id": "E09", "tissue_type": "EcP",          "condition": "endometriosis",   "lesion_site": "peritoneal"},

    # ectopic ovarian
    "GSM6102552_E07_EcO":  {"patient_id": "E07", "tissue_type": "EcO",          "condition": "endometriosis",   "lesion_site": "ovarian"},
    "GSM6102556_E09_EcO":  {"patient_id": "E09", "tissue_type": "EcO",          "condition": "endometriosis",   "lesion_site": "ovarian"},
    "GSM6595261_E10_EcO":  {"patient_id": "E10", "tissue_type": "EcO",          "condition": "endometriosis",   "lesion_site": "ovarian"},
    "GSM6102562_E11_EcO":  {"patient_id": "E11", "tissue_type": "EcO",          "condition": "endometriosis",   "lesion_site": "ovarian"},
}

In [15]:
# -- Collect all wanted files
h5_files = sorted(
    download_dir.rglob(
        "*_filtered_feature_bc_matrix.h5"
    )
)

print(f"Found {len(h5_files)} H5 files.")

for file in h5_files:
    print(file.name)

Found 33 H5 files.
GSM6102532_C01_Ctrl_filtered_feature_bc_matrix.h5
GSM6102533_C02_Ctrl_filtered_feature_bc_matrix.h5
GSM6102534_C03_Ctrl_filtered_feature_bc_matrix.h5
GSM6102537_E01_EuE_filtered_feature_bc_matrix.h5
GSM6102540_E02_EuE_filtered_feature_bc_matrix.h5
GSM6102543_E03_EuE_filtered_feature_bc_matrix.h5
GSM6102546_E04_EuE_filtered_feature_bc_matrix.h5
GSM6102549_E05_EuE_filtered_feature_bc_matrix.h5
GSM6102550_E06_EcP_filtered_feature_bc_matrix.h5
GSM6102551_E06_EuE_filtered_feature_bc_matrix.h5
GSM6102552_E07_EcO_filtered_feature_bc_matrix.h5
GSM6102553_E07_EcP_filtered_feature_bc_matrix.h5
GSM6102554_E07_EuE_filtered_feature_bc_matrix.h5
GSM6102555_E08_EuE_filtered_feature_bc_matrix.h5
GSM6102556_E09_EcO_filtered_feature_bc_matrix.h5
GSM6102560_E09_EuE_filtered_feature_bc_matrix.h5
GSM6102562_E11_EcO_filtered_feature_bc_matrix.h5
GSM6102563_EOR01_filtered_feature_bc_matrix.h5
GSM6102565_EOR03_filtered_feature_bc_matrix.h5
GSM6595248_E01_EcP_filtered_feature_bc_matrix.h5
GS

In [16]:
# -- Set up objects
adatas = []

for file in h5_files:
    sample_id = file.name.replace("_filtered_feature_bc_matrix.h5", "")

    # -- Exclude organoids and unwanted tissue types (Adjacent peritoneum is excluded from v1)

    if any(excl in sample_id for excl in EXCLUDE):
      print(f"Skipping: {sample_id}")
      continue

    # -- Exclude samples without any metadata

    if sample_id not in metadata:
      print(
          f"Skipping {sample_id} - No metadata found."
          )
      continue

    # -- Load data
    adata = sc.read_10x_h5(file)
    adata.var_names_make_unique()

    # -- Add metadata to the object
    meta = metadata[sample_id]
    adata.obs["sample_id"] = sample_id
    adata.obs["patient_id"] = meta['patient_id']
    adata.obs["tissue_type"] = meta["tissue_type"]
    adata.obs["condition"] = meta["condition"]
    adata.obs["lesion_site"] = meta["lesion_site"]
    adata.obs["dataset"] = "GSE179640"

    # -- Append sample name to cell barcodes
    adata.obs_names = [
        f"{sample_id}_{bc}"
        for bc in adata.obs_names
    ]

    print(f"{sample_id} loaded - {adata.n_obs} total cells")
    adatas.append(adata)

/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


GSM6102532_C01_Ctrl loaded - 4133 total cells


/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


GSM6102533_C02_Ctrl loaded - 7801 total cells


/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


...

/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


GSM6595252_E03_EcP loaded - 6691 total cells
Skipping: GSM6595253_E03_EcPA
GSM6595254_E04_EcP loaded - 3536 total cells
Skipping: GSM6595255_E04_EcPA


/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_

GSM6595256_E05_EcP loaded - 4043 total cells
Skipping: GSM6595257_E05_EcPA
GSM6595258_E09_EcP loaded - 2518 total cells
Skipping: GSM6595259_E09_EcPA
Skipping: GSM6595260_E09_EcPA2
GSM6595261_E10_EcO loaded - 12788 total cells


/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [17]:
# -- Check that samples were loaded
if not adatas:
  raise ValueError(
      "No samples were loaded. Check the GEO download directory and file names"
      )

print(f"\n Loaded {len(adatas)} samples")


 Loaded 24 samples


In [18]:
# -- Concat samples
combined = ad.concat(adatas, join = "outer")
combined.var_names_make_unique()

In [19]:
# -- Check concat was successful
print(f"Total cells: {combined.n_obs}")
print(f"Total genes: {combined.n_vars}")

print("\nCells by tissue type:")
print(combined.obs["tissue_type"].value_counts())

print("\nCells by lesion site:")
print(combined.obs["lesion_site"].value_counts(dropna=False))

Total cells: 119560
Total genes: 38224

Cells by tissue type:
tissue_type
EuE     41432
EcP     31664
EcO     28999
Ctrl    17465
Name: count, dtype: int64

Cells by lesion site:
lesion_site
None          58897
peritoneal    31664
ovarian       28999
Name: count, dtype: int64


In [20]:
# -- Check to make sure every donor is accounted for
donor_counts = (combined.obs
.groupby(
    ["tissue_type",
     "patient_id"], observed=True
    )
.size()
.unstack(fill_value=0)
)

donor_counts

patient_id,C01,C02,C03,E01,E02,E03,E04,E05,E06,E07,E08,E09,E10,E11
tissue_type,,,,,,,,,,,,,,
Ctrl,4133,7801,5531,0,0,0,0,0,0,0,0,0,0,0
EcO,0,0,0,0,0,0,0,0,0,2688,0,6152,12788,7371
EcP,0,0,0,2961,5049,6691,3536,4043,2494,4372,0,2518,0,0
EuE,0,0,0,7665,5277,5302,3819,3030,5226,3078,2780,5255,0,0


In [21]:
# -- Save to drive
output_file = (
    interim_data_dir
    / "raw.h5ad"
)

combined.write_h5ad(output_file)

In [22]:
## -- Run to see session info
session_info.show()

/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  mod_version = _find_version(mod.__version__)
/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  mod_version = _find_version(mod.__version__)
